# 03 | LDA classification, three-cluster K-means, and feature ranking

**Purpose.** Analyze well-level morphological profiles from Notebook 02 using LDA, visualize treatment/time separation, explore three-cluster K-means in LDA space, and rank the 30 largest absolute discriminant coefficients.

**Confirmed choices:** three K-means clusters; top 30 features. LDA and clustering use the treatment/time labels defined below.

**Interpretation caveat:** K-means on LDA coordinates is exploratory; LDA is supervised, so cluster correspondence is not an independent validation of class prediction.

In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

INPUT_CSV = Path("../results/INPUT_WELL_PROFILES.csv")  # Replace with Notebook 02 output.
OUTPUT_DIR = Path("../results/lda"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_CLUSTERS = 3
TOP_N = 30
GROUP_COL = "Metadata_compound_time"
PALETTE = {
    "Non-treated_1h": "#272323",
    "AuNP_1h": "#1730F0", "AuNP-and-healthy-serum_1h": "#f79d7c",
    "AuNP-cancer-and-cachexia-serum_1h": "#f26f6f",
    "AuNP_4h": "#0a17a4", "AuNP-and-healthy-serum_4h": "#ef5946",
    "AuNP-cancer-and-cachexia-serum_4h": "#f03b53",
    "AuNP_24h": "#08024C", "AuNP-and-healthy-serum_24h": "#EC3705",
    "AuNP-cancer-and-cachexia-serum_24h": "#FC0509",
}

## 1. Prepare experimental labels and remove unused control time points

The source CSV uses `Metadata_concentration_uM` as a time identifier; it is renamed to `Metadata_time` exactly as in the original analysis. Treatment and time are concatenated to define the LDA classes. Untreated 4 h and 24 h observations are excluded, retaining the untreated 1 h reference.

In [ ]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Set INPUT_CSV to the actual output of Notebook 02: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
if "Metadata_time" not in df and "Metadata_concentration_uM" in df:
    df = df.rename(columns={"Metadata_concentration_uM": "Metadata_time"})
if GROUP_COL not in df:
    df[GROUP_COL] = df["Metadata_compound"].astype(str) + "_" + df["Metadata_time"].astype(str)
df = df.loc[~df[GROUP_COL].isin(["Non-treated_4h", "Non-treated_24h"])].copy()
print(df[GROUP_COL].value_counts().to_string())
unknown_groups = set(df[GROUP_COL]) - set(PALETTE)
if unknown_groups:
    print(f"Groups without predefined colors: {sorted(unknown_groups)}")

## 2. Standardize numerical features and fit LDA

The input features were previously normalized to negative-control wells. Here `StandardScaler` additionally centers and scales each numerical feature across the analyzed observations before fitting the supervised LDA model. Metadata fields and `Metadata_cell_count` are excluded from the feature matrix to avoid using sample identifiers or cell counts as predictors.

In [ ]:
metadata_cols = [col for col in df.columns if col.startswith("Metadata_") or col.startswith("Unnamed:")]
X = df.drop(columns=metadata_cols).select_dtypes(include="number")
X = X.replace([np.inf, -np.inf], np.nan)
if X.empty or X.isna().any().any():
    raise ValueError("Feature matrix is empty or contains NaN/inf. Check the normalized input and feature filtering.")
y = df[GROUP_COL].astype(str)
X_scaled = StandardScaler().fit_transform(X)
n_components = min(X_scaled.shape[1], y.nunique() - 1)
lda = LinearDiscriminantAnalysis(n_components=n_components)
X_lda = lda.fit_transform(X_scaled, y)
lda_df = pd.DataFrame(X_lda, columns=[f"LDA{i+1}" for i in range(X_lda.shape[1])])
lda_df["Class"] = y.to_numpy()
print(f"Input features: {X.shape[1]}; LDA components: {n_components}")

## 3. Visualize LDA component combinations

Each available pair of discriminant components is plotted using the original treatment/time color palette. Export both SVG (editable vector graphics) and PNG (300 dpi). The LDA projection is an in-sample visualization, not a cross-validated prediction.

In [ ]:
components = [col for col in lda_df.columns if col.startswith("LDA")]
pairs = list(itertools.combinations(components, 2))
fig, axes = plt.subplots((len(pairs)+1)//2, 2, figsize=(12, 4*((len(pairs)+1)//2)), squeeze=False)
for ax, (x_name, y_name) in zip(axes.flat, pairs):
    sns.scatterplot(data=lda_df, x=x_name, y=y_name, hue="Class", palette=PALETTE,
                    s=65, ax=ax, legend=False)
    ax.axhline(0, color="gray", linestyle="--", linewidth=.8)
    ax.axvline(0, color="gray", linestyle="--", linewidth=.8)
    ax.set_title(f"{x_name} vs {y_name}")
for ax in list(axes.flat)[len(pairs):]: ax.remove()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "lda_component_pairs.svg", bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "lda_component_pairs.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Explore cluster number using elbow and silhouette scores

For consistency with the original plotting workflow, candidate K-means solutions are evaluated on the first two LDA coordinates. The inertia (within-cluster sum of squares) and silhouette coefficient are shown as descriptive diagnostics. The final number of clusters is fixed at **three**, as used in the study.

In [ ]:
if len(components) < 2: raise ValueError("At least two LDA components are required for clustering plots.")
cluster_X = lda_df[["LDA1", "LDA2"]].to_numpy()
max_k = min(10, len(cluster_X) - 1)
k_values = list(range(2, max_k+1))
inertia, silhouette = [], []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = model.fit_predict(cluster_X)
    inertia.append(model.inertia_)
    silhouette.append(silhouette_score(cluster_X, labels))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(k_values, inertia, marker="o"); axes[0].set(xlabel="Number of clusters", ylabel="Inertia", title="Elbow method")
axes[1].plot(k_values, silhouette, marker="o"); axes[1].set(xlabel="Number of clusters", ylabel="Silhouette score", title="Silhouette analysis")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "kmeans_diagnostics.svg", bbox_inches="tight")
plt.show()

## 5. Fit three-cluster K-means and compare clusters with treatment classes

K-means is fitted to LDA1–LDA2 with `random_state=42`, retaining the three-cluster choice. A cross-tabulation reports the number of wells from each treatment/time class assigned to each cluster. This is a **cluster–class contingency matrix**, rather than a conventional classifier confusion matrix, because cluster identifiers are arbitrary.

In [ ]:
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
lda_df["Cluster"] = kmeans.fit_predict(cluster_X)
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=lda_df, x="LDA1", y="LDA2", hue="Cluster", palette="deep", s=90, ax=ax)
ax.set_title("K-means clustering in LDA space (k = 3)")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "kmeans_3_clusters.svg", bbox_inches="tight"); plt.show()
cluster_table = pd.crosstab(lda_df["Class"], lda_df["Cluster"])
cluster_table.to_csv(OUTPUT_DIR / "cluster_class_contingency.csv")
fig, ax = plt.subplots(figsize=(8, max(5, len(cluster_table)*.4)))
sns.heatmap(cluster_table, annot=True, fmt="d", cmap="Greys", ax=ax)
ax.set(xlabel="K-means cluster", ylabel="Treatment and time", title="Treatment–cluster correspondence")
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "cluster_class_contingency.svg", bbox_inches="tight"); plt.show()

## 6. Rank the 30 largest absolute LDA coefficients per component

The `scalings_` coefficients are used to rank features separately for each LDA component. Their absolute values describe the magnitude of coefficients within the fitted LDA model, not an independent estimate of biological effect size. Export the 30 highest-ranked features for each component as CSV and SVG.

In [ ]:
loadings = pd.DataFrame(lda.scalings_[:, :len(components)], index=X.columns, columns=components)
for component in components:
    top = loadings[component].abs().nlargest(TOP_N)
    top.rename("Absolute LDA coefficient").to_csv(OUTPUT_DIR / f"top_{TOP_N}_{component}.csv")
    fig, ax = plt.subplots(figsize=(9, max(5, len(top)*.25)))
    top.sort_values().plot.barh(ax=ax, color="#607d8b")
    ax.set(xlabel="Absolute LDA coefficient", ylabel="Morphological feature", title=f"Top {TOP_N} features — {component}")
    fig.tight_layout(); fig.savefig(OUTPUT_DIR / f"top_{TOP_N}_{component}.svg", bbox_inches="tight"); plt.show()
lda_df.to_csv(OUTPUT_DIR / "lda_scores_and_clusters.csv", index=False)